# Standalone TA-RecMind Web Demo On Colab

Notebook này chạy độc lập trên Google Colab, không cần upload cả thư mục repo/code lên Drive.

Yêu cầu duy nhất ngoài dữ liệu Hugging Face là file checkpoint model `.pth` sau training. Mặc định notebook đọc checkpoint ở:

`/content/drive/MyDrive/tarecmindV2/weights/tarecmind_amazon_2023_electronics_full_20260525_degree_prior_gate_v2_best.pth`

Lưu ý kỹ thuật: checkpoint `.pth` không chứa sẵn full final graph representations. Để không cần cache embedding riêng, notebook này dựng web demo nhanh bằng learned ID embeddings + text projection của model, với item/user metadata lấy từ Hugging Face. Đây là bản demo tương tác nhẹ trên Colab. Muốn full exact LightGCN inference như notebook training thì cần thêm LLM embedding cache hoặc export final user/item vectors thành inference bundle.

In [ ]:
!pip -q install "gradio>=4.44.0" "sentence-transformers>=3.0.0" "huggingface-hub>=0.23.0" "pyarrow>=14.0.0" "pandas>=2.0.0" "numpy>=1.24.0"

In [ ]:
from google.colab import drive, files
from pathlib import Path
import os

drive.mount('/content/drive')

# Nếu checkpoint nằm ở nơi khác, sửa biến này.
MODEL_PATH = Path('/content/drive/MyDrive/tarecmindV2/weights/tarecmind_amazon_2023_electronics_full_20260525_degree_prior_gate_v2_best.pth')

# Nếu chưa có checkpoint trên Drive, notebook sẽ cho upload một file .pth.
if not MODEL_PATH.exists():
    print('Không tìm thấy checkpoint mặc định. Upload file .pth model:')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('Chưa upload checkpoint .pth')
    MODEL_PATH = Path('/content') / next(iter(uploaded.keys()))

print('MODEL_PATH =', MODEL_PATH)

In [ ]:
import gc, json, math, time
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

GOLD_REPO = 'chuongdo1104/amazon-2023-gold'
SILVER_REPO = 'chuongdo1104/amazon-2023-silver'
TEXT_ENCODER_NAME = 'all-MiniLM-L6-v2'
EMBED_DIM = 128
LLM_DIM = 384
GCN_LAYERS = 2

# Giảm MAX_ITEMS nếu Colab RAM thấp hoặc muốn build nhanh hơn.
MAX_USERS = 5000
MAX_ITEMS = 30000
ENCODE_BATCH = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('DEVICE =', DEVICE)
print('GOLD_REPO =', GOLD_REPO)
print('SILVER_REPO =', SILVER_REPO)

In [ ]:
class TARecMindV2(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=128, llm_dim=384, gcn_layers=2):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embed_dim = embed_dim
        self.gcn_layers = gcn_layers

        self.user_id_emb = nn.Embedding(num_users, embed_dim)
        self.item_id_emb = nn.Embedding(num_items, embed_dim)
        self.text_prj = nn.Linear(llm_dim, embed_dim)
        self.gate_mlp = nn.Sequential(
            nn.Linear(embed_dim * 2 + 1, embed_dim // 2),
            nn.ReLU(),
            nn.Linear(embed_dim // 2, 1),
        )
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def project_text(self, llm_emb):
        return self.text_prj(llm_emb.float())

    def get_final_repr(self, z_g, z_l):
        alpha = torch.sigmoid(self.alpha)
        return alpha * z_g + (1.0 - alpha) * z_l


def hf_download(filename, repo_id=GOLD_REPO):
    return hf_hub_download(repo_id=repo_id, filename=filename, repo_type='dataset')


def load_checkpoint(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)


stats_path = hf_download('gold/gold_dataset_stats.json')
with open(stats_path, 'r', encoding='utf-8') as f:
    stats = json.load(f)

num_users = int(stats['n_users'])
num_items = int(stats['n_items'])
print(f'Gold stats: {num_users:,} users | {num_items:,} items')

ckpt = load_checkpoint(MODEL_PATH)
model = TARecMindV2(num_users, num_items, EMBED_DIM, LLM_DIM, GCN_LAYERS).to(DEVICE)
state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
missing, unexpected = model.load_state_dict(state, strict=False)
model.eval()

print('Checkpoint epoch:', ckpt.get('epoch') if isinstance(ckpt, dict) else None)
print('Missing keys:', missing)
print('Unexpected keys:', unexpected)
print('alpha =', float(torch.sigmoid(model.alpha).detach().cpu()))

In [ ]:
print('Downloading Gold maps from Hugging Face...')
item_map_path = hf_download('gold/gold_item_id_map.parquet')
user_map_path = hf_download('gold/gold_user_id_map.parquet')

df_item_all = pd.read_parquet(
    item_map_path,
    columns=['parent_asin', 'item_idx', 'title', 'main_category', 'popularity_group', 'train_freq'],
)
df_user_all = pd.read_parquet(
    user_map_path,
    columns=['reviewer_id', 'user_idx', 'user_train_freq', 'user_activity_group'],
)

df_item_all['title'] = df_item_all['title'].fillna('').astype(str)
df_item_all['main_category'] = df_item_all['main_category'].fillna('').astype(str)
df_item_all['popularity_group'] = df_item_all['popularity_group'].fillna('COLD_START').astype(str)
df_item_all['train_freq'] = df_item_all['train_freq'].fillna(0).astype('int64')
df_user_all['user_train_freq'] = df_user_all['user_train_freq'].fillna(0).astype('int64')
df_user_all['user_activity_group'] = df_user_all['user_activity_group'].fillna('INACTIVE').astype(str)

print('Items:', len(df_item_all), 'Users:', len(df_user_all))
print(df_item_all['popularity_group'].value_counts(dropna=False).to_string())

In [ ]:
def take_group(df, group, n):
    part = df[df['popularity_group'] == group]
    if len(part) == 0 or n <= 0:
        return part.head(0)
    return part.sort_values(['train_freq', 'item_idx'], ascending=[False, True]).head(n)


warm_items = df_item_all[df_item_all['train_freq'] > 0].copy()
group_quota = {
    'HEAD': int(MAX_ITEMS * 0.30),
    'MID': int(MAX_ITEMS * 0.30),
    'TAIL': int(MAX_ITEMS * 0.40),
}
parts = [take_group(warm_items, group, n) for group, n in group_quota.items()]
df_item = pd.concat(parts, ignore_index=True).drop_duplicates('item_idx')
if len(df_item) < MAX_ITEMS:
    rest = warm_items[~warm_items['item_idx'].isin(df_item['item_idx'])]
    need = MAX_ITEMS - len(df_item)
    df_item = pd.concat([
        df_item,
        rest.sort_values(['train_freq', 'item_idx'], ascending=[False, True]).head(need),
    ], ignore_index=True)
df_item = df_item.sort_values('item_idx').reset_index(drop=True)

df_user = (
    df_user_all.sort_values(['user_train_freq', 'user_idx'], ascending=[False, True])
    .head(MAX_USERS)
    .sort_values('user_idx')
    .reset_index(drop=True)
)

print(f'Demo scope: {len(df_user):,} users | {len(df_item):,} items')
print(df_item['popularity_group'].value_counts(dropna=False).to_string())

In [ ]:
def build_item_text(row):
    title = row.get('title') or '[NO_TITLE]'
    category = row.get('main_category') or '[NO_CATEGORY]'
    group = row.get('popularity_group') or 'UNKNOWN'
    freq = row.get('train_freq') or 0
    return f'title: {title}. category: {category}. popularity_group: {group}. train_frequency: {freq}'[:512]


def build_user_text(row):
    activity = row.get('user_activity_group') or 'INACTIVE'
    freq = row.get('user_train_freq') or 0
    return f'user activity profile: {activity}. train_frequency: {freq}'


print('Loading SentenceTransformer:', TEXT_ENCODER_NAME)
st_model = SentenceTransformer(TEXT_ENCODER_NAME, device=DEVICE)

item_texts = [build_item_text(row) for row in df_item.to_dict('records')]
user_texts = [build_user_text(row) for row in df_user.to_dict('records')]

print('Encoding item texts...')
item_raw = st_model.encode(
    item_texts,
    batch_size=ENCODE_BATCH,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype('float32')

print('Encoding user texts...')
user_raw = st_model.encode(
    user_texts,
    batch_size=ENCODE_BATCH,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True,
).astype('float32')

del st_model
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print('Projecting through trained text_prj and combining with ID embeddings...')
with torch.no_grad():
    alpha = torch.sigmoid(model.alpha)

    item_idx_t = torch.tensor(df_item['item_idx'].to_numpy(), dtype=torch.long, device=DEVICE)
    item_raw_t = torch.from_numpy(item_raw).to(DEVICE)
    item_id = model.item_id_emb(item_idx_t)
    item_txt = model.project_text(item_raw_t)
    item_vectors = F.normalize(model.get_final_repr(item_id, item_txt), p=2, dim=1).cpu().numpy().astype('float32')

    user_idx_t = torch.tensor(df_user['user_idx'].to_numpy(), dtype=torch.long, device=DEVICE)
    user_raw_t = torch.from_numpy(user_raw).to(DEVICE)
    user_id = model.user_id_emb(user_idx_t)
    user_txt = model.project_text(user_raw_t)
    user_vectors = F.normalize(model.get_final_repr(user_id, user_txt), p=2, dim=1).cpu().numpy().astype('float32')

print('Vectors ready:', user_vectors.shape, item_vectors.shape)

In [ ]:
print('Downloading Gold train edge_index for seen-item masking...')
edge_path = hf_download('gold/gold_edge_index.npy')
edge_index = np.load(edge_path, mmap_mode='r')

selected_user_set = set(df_user['user_idx'].astype(int).tolist())
selected_item_set = set(df_item['item_idx'].astype(int).tolist())
seen_by_user = defaultdict(set)

edge_users = np.asarray(edge_index[0])
edge_items = np.asarray(edge_index[1])
mask = np.isin(edge_users, list(selected_user_set)) & np.isin(edge_items, list(selected_item_set))
for u, i in zip(edge_users[mask], edge_items[mask]):
    seen_by_user[int(u)].add(int(i))

user_row_by_idx = {int(v): p for p, v in enumerate(df_user['user_idx'].to_numpy())}
item_pos_by_idx = {int(v): p for p, v in enumerate(df_item['item_idx'].to_numpy())}

print(f'Seen-mask users: {len(seen_by_user):,}')

In [ ]:
import gradio as gr

user_choices = []
for row in df_user.sort_values(['user_train_freq', 'user_idx'], ascending=[False, True]).itertuples(index=False):
    user_choices.append(f'{int(row.user_idx)} | {row.reviewer_id} | freq={int(row.user_train_freq)} | {row.user_activity_group}')


def parse_user_idx(choice):
    if choice is None or str(choice).strip() == '':
        return int(df_user.iloc[0]['user_idx'])
    return int(str(choice).split('|', 1)[0].strip())


def recommend(user_choice, top_k, groups, warm_only, mask_seen, popularity_penalty, category_contains):
    user_idx = parse_user_idx(user_choice)
    if user_idx not in user_row_by_idx:
        return pd.DataFrame([{'message': f'user_idx {user_idx} is not in this demo subset'}])

    user_pos = user_row_by_idx[user_idx]
    scores = item_vectors @ user_vectors[user_pos]
    keep = np.ones(len(df_item), dtype=bool)

    if groups:
        keep &= df_item['popularity_group'].isin(groups).to_numpy()
    if warm_only:
        keep &= df_item['train_freq'].to_numpy() > 0
    if category_contains and str(category_contains).strip():
        q = str(category_contains).strip().lower()
        keep &= df_item['main_category'].str.lower().str.contains(q, regex=False, na=False).to_numpy()
    if mask_seen:
        seen = seen_by_user.get(user_idx, set())
        if seen:
            keep &= ~df_item['item_idx'].isin(seen).to_numpy()
    if popularity_penalty and float(popularity_penalty) > 0:
        scores = scores - float(popularity_penalty) * np.log1p(df_item['train_freq'].to_numpy(dtype='float32'))

    scores = scores.copy()
    scores[~keep] = -np.inf
    available = int(np.isfinite(scores).sum())
    if available == 0:
        return pd.DataFrame([{'message': 'No candidates after filters'}])

    k = max(1, min(int(top_k), available))
    pos = np.argpartition(-scores, k - 1)[:k]
    pos = pos[np.argsort(-scores[pos])]

    out = df_item.iloc[pos][['parent_asin', 'title', 'main_category', 'popularity_group', 'train_freq', 'item_idx']].copy()
    out.insert(0, 'rank', np.arange(1, len(out) + 1))
    out.insert(1, 'score', scores[pos])
    out['score'] = out['score'].map(lambda x: round(float(x), 6))
    return out


with gr.Blocks(title='TA-RecMind Colab Web Demo') as demo:
    gr.Markdown('# TA-RecMind Colab Web Demo')
    gr.Markdown('Standalone checkpoint + Hugging Face demo. This lightweight mode uses trained ID embeddings and text projection; full graph inference needs exported final vectors or LLM caches.')
    with gr.Row():
        user_input = gr.Dropdown(choices=user_choices, value=user_choices[0], label='User', filterable=True)
        top_k = gr.Slider(1, 50, value=20, step=1, label='Top K')
    with gr.Row():
        group_input = gr.CheckboxGroup(['HEAD', 'MID', 'TAIL', 'COLD_START'], value=['HEAD', 'MID', 'TAIL'], label='Candidate groups')
        category_input = gr.Textbox(label='Category contains', placeholder='optional')
    with gr.Row():
        warm_only = gr.Checkbox(value=True, label='Warm items only')
        mask_seen = gr.Checkbox(value=True, label='Hide seen train items')
        pop_penalty = gr.Slider(0.0, 1.0, value=0.0, step=0.01, label='Popularity penalty')
    btn = gr.Button('Recommend', variant='primary')
    output = gr.Dataframe(label='Recommendations', interactive=False, wrap=True)
    btn.click(recommend, [user_input, top_k, group_input, warm_only, mask_seen, pop_penalty, category_input], output)

demo.load(recommend, [user_input, top_k, group_input, warm_only, mask_seen, pop_penalty, category_input], output)

demo.launch(share=True, debug=True)